# Fine-tuning Whisper Bambara (version lisible)

Ce notebook est une version nettoyee et reordonnee du travail experimental.
Il conserve uniquement la partie **fine-tuning Whisper pour la transcription Bambara**.

> Note importante : ce notebook **n'est pas le fichier complet historique** du projet.
> La version brute complete est conservee dans `archive/projetBambara_full_backup_raw.ipynb`.


## 1) Preparation du dataset (JSON + audio)


In [ ]:
import os
import json

# json_dir = "Whisper_data/Whisper_data"
audio_dir = "audios/audios"
json_dir = "whisper_data/whisper_data"

dataset_entries = []

for filename in os.listdir(json_dir):
    if filename.endswith(".json"):
        json_path = os.path.join(json_dir, filename)

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            audio_filename = data["audio_filename"]
            audio_path = os.path.join(audio_dir, f"{audio_filename}.wav")

            for segment in data["transcription"]:
                # ✅ Vérifie que les champs nécessaires existent
                if all(key in segment for key in ["start", "end", "text", "translation"]):
                    entry = {
                        "audio_filepath": audio_path,
                        "start": segment["start"],
                        "end": segment["end"],
                        "text": segment["text"],
                        "translation": segment["translation"]
                    }
                    dataset_entries.append(entry)
                else:
                    print(f"⛔ Segment ignoré dans {filename} car il manque une clé.")

print(f"\n✅ Nombre total de segments extraits : {len(dataset_entries)}")
if dataset_entries:
    print("🔍 Exemple d'entrée :")
    print(dataset_entries[0])



✅ Nombre total de segments extraits : 187
🔍 Exemple d'entrée :
{'audio_filepath': 'audios/audios/Apprendre le Bambara Tranquillement au Lit  200 PhrasesLire ou Écouter ou Répéter  Zanga School_segment_911.wav', 'start': 0.0, 'end': 23.83, 'text': 'Tan in tila ka ɲɛ, nin sira in tila ka ɲɛ, nin ye mun fɛn ye?', 'translation': "Divisez correctement le dix, divisez correctement cette ligne, qu'est-ce que c'est ?"}


In [ ]:
import os
import json


json_path = "whisper_jeli/whisper_jeli/jeli-asr-rmai-train-manifest.json"
audio_dir = "audios/audios"
jeli_entries = []

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())

            if all(key in data for key in ["audio_filepath", "text", "duration"]):
                audio_file = os.path.basename(data["audio_filepath"])  # extrait juste le nom du fichier
                corrected_path = os.path.join(audio_dir, audio_file)   # nouveau chemin

                entry = {
                    "audio_filepath": corrected_path,
                    "start": 0.0,  # ⚠️ on suppose que l'audio commence à 0.0 s
                    "end": data["duration"],
                    "text": data["text"],
                    "translation": ""
                }
                jeli_entries.append(entry)
        except json.JSONDecodeError as e:
            print(f"⛔ JSON invalide : {e}")

# Résumé
print(f"\n✅ Nombre total d'entrées Jeli : {len(jeli_entries)}")
print("🔍 Exemple :", jeli_entries[0] if jeli_entries else "aucune")



✅ Nombre total d'entrées Jeli : 1715
🔍 Exemple : {'audio_filepath': 'audios/audios/griots_r8-646950-653370.wav', 'start': 0.0, 'end': 6.42, 'text': "minun nana kita bolo fɛ o tun y'a ta kɛ woro feere de ye", 'translation': ''}


In [ ]:
import os

# On teste les 5 premiers fichiers (ou moins si tu en as moins)
print("\n🎧 Vérification des fichiers audio :")
for i, entry in enumerate(jeli_entries[:5]):
    path = entry["audio_filepath"]
    if os.path.exists(path):
        print(f"✅ Fichier trouvé : {path}")
    else:
        print(f"❌ Fichier manquant : {path}")



🎧 Vérification des fichiers audio :
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-45480-63500.wav
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-77180-79020.wav
✅ Fichier trouvé : jeliaudio/jeliaudio/griots_r1-111860-116180.wav


In [ ]:
# 🔁 Fusionner les deux
final_dataset = dataset_entries + jeli_entries

# 📊 Résumé
print(f"\n✅ Nombre total d'entrées finales : {len(final_dataset)}")
if final_dataset:
    print("🔍 Exemple d'entrée fusionnée :")
    print(final_dataset[0])



✅ Nombre total d'entrées finales : 1902
🔍 Exemple d'entrée fusionnée :
{'audio_filepath': 'audios/audios/Apprendre le Bambara Tranquillement au Lit  200 PhrasesLire ou Écouter ou Répéter  Zanga School_segment_911.wav', 'start': 0.0, 'end': 23.83, 'text': 'Tan in tila ka ɲɛ, nin sira in tila ka ɲɛ, nin ye mun fɛn ye?', 'translation': "Divisez correctement le dix, divisez correctement cette ligne, qu'est-ce que c'est ?"}


In [ ]:
import os
import torchaudio
from tqdm import tqdm

# Nettoyage des entrées dont les fichiers audio n'existent pas
final_dataset = [entry for entry in final_dataset if os.path.exists(entry["audio_filepath"])]

# Nouveau dossier pour les fichiers audio convertis
resampled_dir = "audios_resampled"
os.makedirs(resampled_dir, exist_ok=True)

resampled_entries = []

print("Rééchantillonnage des fichiers audio...")

for entry in tqdm(final_dataset):
    orig_path = entry["audio_filepath"]

    if not os.path.exists(orig_path):
        print(f"Fichier introuvable : {orig_path}")
        continue

    # Nouveau chemin
    filename = os.path.basename(orig_path)
    new_path = os.path.join(resampled_dir, filename)

    # Chargement + rééchantillonnage
    waveform, sample_rate = torchaudio.load(orig_path)

    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    torchaudio.save(new_path, waveform, 16000)

    # Mise à jour du chemin dans l'entrée
    resampled_entry = entry.copy()
    resampled_entry["audio_filepath"] = new_path
    resampled_entries.append(resampled_entry)

print(f"\nTous les fichiers ont été rééchantillonnés et enregistrés dans {resampled_dir}.")

# Suppression des fichiers audio orphelins dans le dossier "audios"
audio_dir = "audios/audios"
used_files = set(os.path.basename(entry["audio_filepath"]) for entry in final_dataset)

for file in os.listdir(audio_dir):
    if file not in used_files:
        full_path = os.path.join(audio_dir, file)
        os.remove(full_path)
        print(f"Fichier non utilisé supprimé : {full_path}")


Rééchantillonnage des fichiers audio...


100%|██████████| 1811/1811 [00:42<00:00, 42.71it/s]



Tous les fichiers ont été rééchantillonnés et enregistrés dans audios_resampled.
Fichier non utilisé supprimé : audios/audios/409.wav
Fichier non utilisé supprimé : audios/audios/242.mp3
Fichier non utilisé supprimé : audios/audios/342.wav
Fichier non utilisé supprimé : audios/audios/451.wav
Fichier non utilisé supprimé : audios/audios/283.wav
Fichier non utilisé supprimé : audios/audios/135.mp3
Fichier non utilisé supprimé : audios/audios/446.wav
Fichier non utilisé supprimé : audios/audios/242.wav
Fichier non utilisé supprimé : audios/audios/209.mp3
Fichier non utilisé supprimé : audios/audios/135.wav
Fichier non utilisé supprimé : audios/audios/219.mp3
Fichier non utilisé supprimé : audios/audios/444.wav
Fichier non utilisé supprimé : audios/audios/081.mp3
Fichier non utilisé supprimé : audios/audios/346.mp3
Fichier non utilisé supprimé : audios/audios/146.wav
Fichier non utilisé supprimé : audios/audios/126.wav
Fichier non utilisé supprimé : audios/audios/456.mp3
Fichier non utili

In [ ]:
from datasets import Dataset

# Crée un objet Dataset à partir de la liste de dictionnaires
hf_dataset = Dataset.from_list(resampled_entries)

# 🔍 Aperçu
hf_dataset = hf_dataset.shuffle(seed=42)
hf_dataset.select(range(1)).to_pandas()


,audio_filepath,start,end,text,translation
0,audios_resampled/griots_r22-559040-564320.wav,0.0,5.28,n'a ya fili a bɛ se ka cɛ tan ni kɔ la bin,


In [ ]:
from datasets import Audio

# On transforme le chemin en audio réel (auto-chargé à la volée)
hf_dataset = hf_dataset.cast_column("audio_filepath", Audio(sampling_rate=16000))

# Pour que ça soit plus clair, on renomme la colonne
hf_dataset = hf_dataset.rename_column("audio_filepath", "audio")

# Vérifions que tout est bon
hf_dataset[0]


{'audio': {'path': 'audios_resampled/griots_r22-559040-564320.wav',
  'array': array([ 6.10351562e-05,  6.10351562e-05,  1.83105469e-04, ...,
         -1.83105469e-04, -3.05175781e-05, -1.52587891e-04]),
  'sampling_rate': 16000},
 'start': 0.0,
 'end': 5.28,
 'text': "n'a ya fili a bɛ se ka cɛ tan ni kɔ la bin",
 'translation': ''}

In [ ]:
# from transformers import WhisperProcessor

# # Exemple avec le modèle base (tu peux choisir un autre)
# processor = WhisperProcessor.from_pretrained("openai/whisper-base")



preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

## 2) Preprocessing Whisper (features + labels)


In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    # Prétraitement avec le processor Whisper
    inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["text"],
    )

    # Ajoute la longueur pour filtrage plus tard
    inputs["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return inputs


In [ ]:
hf_dataset = hf_dataset.map(
    prepare_dataset,
    remove_columns=hf_dataset.column_names,  # on garde que les colonnes utiles après transformation
    num_proc=1,
)


Map:   0%|          | 0/1811 [00:00<?, ? examples/s]

In [ ]:
max_input_length = 30.0

# hf_dataset = hf_dataset.filter(
#     lambda x: x["input_length"] < max_input_length,
#     input_columns=["input_length"],
# )
hf_dataset = hf_dataset.filter(
    lambda x: x < max_input_length,
    input_columns=["input_length"]
)


Filter:   0%|          | 0/1811 [00:00<?, ? examples/s]

In [ ]:
hf_dataset[0].keys()
# Tu verras normalement : ['input_features', 'labels', 'input_length']


dict_keys(['input_features', 'labels', 'input_length'])

## 3) Data collator et metriques WER


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"][0]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


In [ ]:
# 2. Activer le retour de l'attention mask
processor.feature_extractor.return_attention_mask = True
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token  # par défaut

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(hf_dataset, batch_size=2, collate_fn=data_collator)

batch = next(iter(dataloader))
batch.keys()  # devrait contenir 'input_features' et 'labels'


dict_keys(['input_features', 'attention_mask', 'labels'])

In [ ]:
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Métrique d'évaluation : Word Error Rate (WER)
metric = evaluate.load("wer")

# Pour "nettoyer" les textes avant de calculer le WER
normalizer = BasicTextNormalizer()


In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Remplacer les -100 (tokens ignorés) par le token de padding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Décodage brut (texte sans tokens spéciaux)
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Word Error Rate sans normalisation
    wer_ortho = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Word Error Rate avec normalisation
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]

    # Supprimer les labels vides (problème courant)
    pred_str_norm = [p for i, p in enumerate(pred_str_norm) if len(label_str_norm[i]) > 0]
    label_str_norm = [l for l in label_str_norm if len(l) > 0]

    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}


## 4) Chargement du modele et configuration d'entrainement


In [ ]:
import torch
from transformers import WhisperForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📦 Modèle chargé sur :", device)

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# 💡 Corrige le warning ici :
model.config.forced_decoder_ids = None

model.to(device)


📦 Modèle chargé sur : cuda


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [ ]:
from functools import partial

# Empêche une erreur lors de l'entraînement avec checkpointing
model.config.use_cache = False

# Fixer les options de génération
model.generate = partial(
    model.generate,
    language="french",  # 🔁 remplace ici par "french" si c'est ta langue cible
    task="transcribe",   # ou "transcribe" si c'est de la transcription directe
    use_cache=True
)


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bambara",  # nom du répertoire de sortie
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    fp16_full_eval=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
from datasets import DatasetDict

# Diviser le dataset en 80% pour l'entraînement et 20% pour le test
hf_dataset_split = hf_dataset.train_test_split(test_size=0.2)

# Maintenant, hf_dataset_split contient "train" et "test"
train_dataset = hf_dataset_split["train"]
test_dataset = hf_dataset_split["test"]


## 5) Entrainement et evaluation


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,  # Ensemble d'entraînement
    eval_dataset=test_dataset,  # Ensemble de test
    data_collator=data_collator,  # Le collateur de données
    compute_metrics=compute_metrics,  # Calcul des métriques (par exemple, WER)
    tokenizer=processor,  # Le processor Whisper
)


<ipython-input-38-c3075a680a57>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer Ortho,Wer
1,No log,2.780564,363.750000,359.248555
2,No log,1.589950,179.375000,230.635838
3,2.265000,1.196536,141.562500,144.508671
4,2.265000,1.027056,111.875000,107.225434
5,0.677900,0.972821,105.625000,102.601156
6,0.677900,0.963565,105.312500,101.156069
7,0.677900,0.978302,109.687500,106.358382
8,0.193100,0.963333,102.812500,99.710983
9,0.193100,0.966830,111.250000,105.202312
10,0.066900,0.958066,110.000000,104.624277


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=100, training_loss=0.8006967449188233, metrics={'train_runtime': 476.434, 'train_samples_per_second': 0.819, 'train_steps_per_second': 0.21, 'total_flos': 1.125483061248e+17, 'train_loss': 0.8006967449188233, 'epoch': 10.0})